# freqgen — SPAI Evasion with Scene-Paired RAISE-1k (Kaggle)

**Version 2** — Uses RAISE-1k RAW camera images as real baseline instead of COCO.

Why this matters:
- Synthbuster SD1.4 fakes were generated from RAISE-1k scene prompts
- Each fake has a scene-paired real: `r00c4a543t.png` (fake) ↔ `r00c4a543t.TIF` (real)
- RAISE-1k RAW images have higher HF energy than COCO (no JPEG compression)
- This reproduces the expected ~21x high-band deficit direction
- Removes the COCO circular-reasoning caveat from Version 1

**Enable GPU:** Settings (gear icon, top right) → Accelerator → GPU P100 or T4

Reference: SPAI — arXiv 2411.19417, code github.com/mever-team/spai

## 1. Install SPAI

In [ ]:
import os

if not os.path.exists('/kaggle/working/spai'):
    !git clone https://github.com/mever-team/spai.git /kaggle/working/spai -q
%cd /kaggle/working/spai

!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r requirements.txt filetype -q

import numpy as np, torch
print(f'numpy {np.__version__}  torch {torch.__version__}  cuda={torch.cuda.is_available()}')
print('SPAI install OK')

## 2. Download SPAI weights

In [ ]:
import os
os.makedirs('/kaggle/working/spai/weights', exist_ok=True)
if not os.path.exists('/kaggle/working/spai/weights/spai.pth'):
    !pip install gdown -q
    !gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O /kaggle/working/spai/weights/spai.pth
sz = os.path.getsize('/kaggle/working/spai/weights/spai.pth') // 1_000_000
print(f'Weights: {sz} MB  (expect ~934)')

## 3. Download data

- **Synthbuster SD1.4** (~12 GB from Zenodo) — real SD1.4 fakes
- **RAISE-1k index** (CSV, 528 KB from GitHub) — maps filenames to TIFF download URLs
- **RAISE-1k TIFFs** (~35-50 MB each, download only the N scene-paired ones)

Scene pairing: for each Synthbuster fake `r00c4a543t.png`, we download `r00c4a543t.TIF` from RAISE.

In [ ]:
import os, glob, csv, urllib.request
from tqdm.notebook import tqdm

BASE = '/kaggle/working'
N = 30  # number of scene-paired triplets (real, fake, matched)

# --- Synthbuster SD1.4 (fakes) ---
SYNTH = f'{BASE}/synthbuster/stable-diffusion-1-4'
if not os.path.exists(SYNTH) or len(glob.glob(f'{SYNTH}/*.png')) < 100:
    print('Downloading Synthbuster SD1.4 (~12 GB, ~15 min)...')
    !wget -L -c 'https://zenodo.org/records/10066460/files/synthbuster.zip' -O /tmp/sb.zip
    sz_mb = os.path.getsize('/tmp/sb.zip') // 1_000_000
    print(f'Downloaded: {sz_mb} MB')
    if sz_mb < 100:
        raise RuntimeError(f'Synthbuster download failed ({sz_mb} MB).')
    os.makedirs(os.path.dirname(SYNTH), exist_ok=True)
    !unzip -q /tmp/sb.zip 'synthbuster/stable-diffusion-1-4/*' -d {BASE}
    os.remove('/tmp/sb.zip')

fake_paths = sorted(glob.glob(f'{SYNTH}/*.png'))[:N]
print(f'Fakes: {len(fake_paths)} SD1.4 images')
print(f'  Example filename: {os.path.basename(fake_paths[0])}')

# --- RAISE-1k index (CSV with download URLs) ---
print('\nDownloading RAISE-1k index CSV...')
!wget -q 'https://raw.githubusercontent.com/zarifbinhasnat/freqgen/main/data/raise_1k_index.csv' -O /tmp/raise_index.csv

url_map = {}
with open('/tmp/raise_index.csv') as f:
    for row in csv.DictReader(f):
        url_map[row['File']] = row['TIFF']
print(f'RAISE index loaded: {len(url_map)} entries')

# --- RAISE-1k TIFFs (scene-paired with our N fakes) ---
RAISE = f'{BASE}/raise_1k'
os.makedirs(RAISE, exist_ok=True)

synth_ids = [os.path.splitext(os.path.basename(p))[0] for p in fake_paths]
missing_in_raise = [fid for fid in synth_ids if fid not in url_map]
if missing_in_raise:
    print(f'WARNING: {len(missing_in_raise)} fakes not in RAISE index: {missing_in_raise[:3]}')

print(f'\nDownloading {N} scene-paired RAISE-1k TIFFs (~35-50 MB each)...')
downloaded, skipped = 0, 0
real_paths = []
for fid in tqdm(synth_ids):
    out = f'{RAISE}/{fid}.TIF'
    real_paths.append(out)
    if os.path.exists(out):
        skipped += 1
        continue
    url = url_map.get(fid)
    if not url:
        print(f'  SKIP (not in RAISE): {fid}')
        continue
    try:
        urllib.request.urlretrieve(url, out)
        downloaded += 1
    except Exception as e:
        print(f'  FAIL {fid}: {e}')

real_paths = [p for p in real_paths if os.path.exists(p)]
total_mb = sum(os.path.getsize(p) for p in real_paths) // 1_000_000
print(f'RAISE-1k: {len(real_paths)} TIFFs ready ({total_mb} MB total), {skipped} cached')

## 4. Spectral matching attack

Build real spectral target from RAISE-1k TIFFs (scene-paired).
For each fake, rescale its Fourier magnitude to match the target.
Phase (structure/content) is preserved exactly.

In [ ]:
import numpy as np, glob, os
from PIL import Image
from tqdm.notebook import tqdm

SIZE = 256
BASE = '/kaggle/working'
SYNTH = f'{BASE}/synthbuster/stable-diffusion-1-4'
RAISE = f'{BASE}/raise_1k'

fake_paths = sorted(glob.glob(f'{SYNTH}/*.png'))[:N]
real_paths = [f'{RAISE}/{os.path.splitext(os.path.basename(p))[0]}.TIF' for p in fake_paths]
real_paths = [p for p in real_paths if os.path.exists(p)]
# Trim fake_paths to match available real
paired_ids = {os.path.splitext(os.path.basename(p))[0] for p in real_paths}
fake_paths = [p for p in fake_paths if os.path.splitext(os.path.basename(p))[0] in paired_ids]
print(f'Scene-paired: {len(real_paths)} real RAISE / {len(fake_paths)} fake SD1.4')

def load_gray(p):
    return np.array(Image.open(p).convert('L').resize((SIZE, SIZE)), dtype=np.float64)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch))
    cy, cx = ch.shape[0]//2, ch.shape[1]//2
    y, x = np.ogrid[:ch.shape[0], :ch.shape[1]]
    r = np.round(np.sqrt((y-cy)**2 + (x-cx)**2)).astype(int)
    mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=np.abs(F).ravel())
    c = np.bincount(r.ravel())
    return t[:mr] / np.maximum(c[:mr], 1)

def spectral_match(img, target, gain_clip=(0.1, 12.0)):
    F = np.fft.fftshift(np.fft.fft2(img))
    cy, cx = img.shape[0]//2, img.shape[1]//2
    y, x = np.ogrid[:img.shape[0], :img.shape[1]]
    r = np.round(np.sqrt((y-cy)**2 + (x-cx)**2)).astype(int)
    mr = len(target)
    gain = np.clip(target / (radial_profile(img) + 1e-12), *gain_clip)
    gain[0] = 1.0
    gmap = gain[np.clip(r, 0, mr-1)]
    gmap[r >= mr] = 1.0
    return np.clip(np.fft.ifft2(np.fft.ifftshift(F * gmap)).real, 0, 255)

print('Building real spectral target from RAISE-1k TIFFs...')
target = np.mean([radial_profile(load_gray(p)) for p in tqdm(real_paths)], axis=0)

MATCHED = f'{BASE}/matched_sd14_raise'
os.makedirs(MATCHED, exist_ok=True)
matched_paths = []
print('Running spectral matching attack...')
for p in tqdm(fake_paths):
    m = spectral_match(load_gray(p), target)
    out = f'{MATCHED}/{os.path.basename(p)}'
    Image.fromarray(m.astype(np.uint8)).save(out)
    matched_paths.append(out)
print(f'Saved {len(matched_paths)} matched fakes')

rh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in real_paths])
fh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in fake_paths])
mh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in matched_paths])
print(f'\n=== SPECTRAL GAP (RAISE-1k baseline) ===')
print(f'real (RAISE-1k) = {rh:.1f}  fake (SD1.4) = {fh:.1f}  matched = {mh:.1f}')
print(f'Gap real/fake   = {rh/fh:.2f}x   (expect >1: RAISE has MORE HF than SD1.4)')
print(f'Gap real/matched= {rh/mh:.2f}x   (expect ~1: attack closes the gap)')

## 5. Prepare directories for SPAI

In [ ]:
import os, shutil, glob
BASE = '/kaggle/working'
SYNTH = f'{BASE}/synthbuster/stable-diffusion-1-4'
RAISE = f'{BASE}/raise_1k'
MATCHED = f'{BASE}/matched_sd14_raise'

N = 30
fake_paths = sorted(glob.glob(f'{SYNTH}/*.png'))[:N]
real_paths = [f'{RAISE}/{os.path.splitext(os.path.basename(p))[0]}.TIF'
              for p in fake_paths if os.path.exists(
                  f'{RAISE}/{os.path.splitext(os.path.basename(p))[0]}.TIF')]
matched_paths = sorted(glob.glob(f'{MATCHED}/*.png'))[:N]

SPAI_IN = f'{BASE}/spai_input_raise'
for tag, paths in [
    ('real',    real_paths),
    ('fake',    fake_paths),
    ('matched', matched_paths),
]:
    d = f'{SPAI_IN}/{tag}'
    os.makedirs(d, exist_ok=True)
    for p in paths:
        dst = f'{d}/{os.path.basename(p)}'
        if not os.path.exists(dst):
            shutil.copy2(p, dst)
    print(f'{tag}: {len(os.listdir(d))} images')

## 6. SPAI inference

Run from `/kaggle/working/spai` so `./weights/` and `./configs/` resolve.

In [ ]:
import os
BASE = '/kaggle/working'
SPAI_IN = f'{BASE}/spai_input_raise'

for tag in ['real', 'fake', 'matched']:
    os.makedirs(f'{BASE}/spai_output_raise/{tag}', exist_ok=True)

%cd /kaggle/working/spai

print('SPAI on REAL images (RAISE-1k TIFFs)...')
!python -m spai infer \
    --input {SPAI_IN}/real \
    --output {BASE}/spai_output_raise/real

print('SPAI on FAKE images (Synthbuster SD1.4)...')
!python -m spai infer \
    --input {SPAI_IN}/fake \
    --output {BASE}/spai_output_raise/fake

print('SPAI on MATCHED (attacked) fakes...')
!python -m spai infer \
    --input {SPAI_IN}/matched \
    --output {BASE}/spai_output_raise/matched

print('SPAI inference done')

## 7. Results — the evasion table (RAISE-1k baseline)

In [ ]:
import pandas as pd, glob, numpy as np
BASE = '/kaggle/working'

def load_scores(tag):
    csvs = glob.glob(f'{BASE}/spai_output_raise/{tag}/**/*.csv', recursive=True)
    if not csvs:
        raise FileNotFoundError(f'No SPAI output for {tag}')
    df = pd.read_csv(csvs[0])
    print(f'{tag}: {len(df)} rows, columns: {list(df.columns)}')
    # Print first few rows with raw scores
    print(df[['image', 'spai']].head(3).to_string(index=False))
    return df

res_real    = load_scores('real')
res_fake    = load_scores('fake')
res_matched = load_scores('matched')

score_col = 'spai'

fake_det    = (res_fake[score_col]    >= 0.5).mean()
matched_det = (res_matched[score_col] >= 0.5).mean()
real_fp     = (res_real[score_col]    >= 0.5).mean()

# Recompute spectral gaps (reload in case cell 4 vars are gone)
from PIL import Image
def load_gray(p, sz=256):
    return np.array(Image.open(p).convert('L').resize((sz,sz)), dtype=np.float64)
def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch))
    cy, cx = ch.shape[0]//2, ch.shape[1]//2
    y, x = np.ogrid[:ch.shape[0], :ch.shape[1]]
    r = np.round(np.sqrt((y-cy)**2 + (x-cx)**2)).astype(int)
    mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=np.abs(F).ravel())
    c = np.bincount(r.ravel())
    return t[:mr] / np.maximum(c[:mr], 1)

SYNTH   = f'{BASE}/synthbuster/stable-diffusion-1-4'
RAISE   = f'{BASE}/raise_1k'
MATCHED = f'{BASE}/matched_sd14_raise'
import os
N = 30
fake_p    = sorted(glob.glob(f'{SYNTH}/*.png'))[:N]
real_p    = [f'{RAISE}/{os.path.splitext(os.path.basename(p))[0]}.TIF' for p in fake_p]
real_p    = [p for p in real_p if os.path.exists(p)]
matched_p = sorted(glob.glob(f'{MATCHED}/*.png'))[:N]

rh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in real_p])
fh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in fake_p])
mh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in matched_p])

print()
print('='*60)
print('freqgen — SPAI Evasion Table (Synthbuster SD1.4 + RAISE-1k)')
print('='*60)
print(f'Real baseline: RAISE-1k RAW TIFFs (scene-paired with fakes)')
print(f'Spectral gap  real/fake    = {rh/fh:.2f}x  (RAISE has more HF than SD1.4)')
print(f'Spectral gap  real/matched = {rh/mh:.2f}x  (attack closes the gap)')
print()
print(f'SPAI detects raw SD1.4 fakes:      {fake_det:.0%}')
print(f'SPAI detects matched fakes:        {matched_det:.0%}')
print(f'SPAI false-positive on RAISE-1k:   {real_fp:.0%}')
print(f'Evasion rate (spectral attack):    {1-matched_det:.0%}')
print('='*60)
if 1-matched_det > 0.5:
    print('RESULT: Attack EVADES SPAI -> gap found in CVPR 2025 SOTA')
else:
    print('RESULT: SPAI survives attack -> investigate why')
print()
print('Score distributions:')
print(f'  Real (RAISE-1k) mean={res_real[score_col].mean():.4f}  std={res_real[score_col].std():.4f}')
print(f'  Fake (SD1.4)    mean={res_fake[score_col].mean():.4f}  std={res_fake[score_col].std():.4f}')
print(f'  Matched         mean={res_matched[score_col].mean():.4f}  std={res_matched[score_col].std():.4f}')
print()
print('Raw score ranges:')
print(f'  Fake    min={res_fake[score_col].min():.6e}  max={res_fake[score_col].max():.6e}')
print(f'  Matched min={res_matched[score_col].min():.6e}  max={res_matched[score_col].max():.6e}')